# Chunk 26 — Frontier Scaleup: Investigating Data Scale vs. Asymmetric Loss

**Project:** GATv2 Surrogate Model for Pixelated Microstrip Patch Antennas  
**Goal:** Determine whether scaling the training set from 2,000 to 4,000 samples alters the Pareto frontier between $S_{11}$ spectrum MAE and deep-resonance False Negative Rate (FNR).

### Core Questions Addressed
1. **Does increasing training data from 2,000 to 4,000 samples reduce deep-resonance under-prediction on its own under standard symmetric $L_2$ loss?** (Training-budget effect vs. loss geometry effect).
2. **Does the selected asymmetric loss ($\kappa = \kappa^*$) retain its depth-bias mitigation advantage over $\kappa = 1.0$ at 4,000 samples?** (Small-sample artifact vs. scale-invariant structural fix).
3. **Does data scale alter the accuracy tradeoff ($\Delta\text{MAE}\%$) incurred by the asymmetric penalty?**

### Experimental Protocol
- **Arms:** 2 arms ($\kappa = 1.0$ baseline anchor vs. $\kappa = \kappa^*$ selected in Chunk 25).
- **Training Subset:** Grid-stratified 4,000-sample subset drawn from `finetune_pool_indices.json` with `seed=2026` (saved to `DATA_ROOT/artifacts/tradeoff_subset_4000.json`).
- **Seeds:** 5 random seeds $[42, 43, 44, 45, 46]$ for each arm ($2 \times 5 = 10$ training runs).
- **Validation Set:** Identical 1,496 validation samples (`finetune_val_indices.json`) — validation set does not change with training subset size.
- **Cross-Scale Comparison:** Side-by-side comparison of 2,000-sample results (from Chunk 25) vs. 4,000-sample results (this chunk) with apples-to-apples baseline degradation recomputation and formal 3-way interpretation.


In [ ]:
# Install required packages
# Use prebuilt wheels for torch-scatter/torch-sparse to avoid slow source builds
import torch
tv = torch.__version__.split('+')[0]   # e.g. '2.6.0'
cv = torch.version.cuda.replace('.', '')  # e.g. '124'
whl = f'https://data.pyg.org/whl/torch-{tv}+cu{cv}.html'
print(f'PyG wheel index: {whl}')
!pip install -q torch-scatter torch-sparse -f {whl}
!pip install -q torch-geometric scipy pandas matplotlib seaborn tqdm pyarrow fastparquet scikit-learn


In [ ]:
# Clone the repository and add src to sys.path
import os
import sys

REPO_ROOT = '/content/antenna-gnn'

if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull

if f'{REPO_ROOT}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_ROOT}/src')


In [ ]:
# Mount Google Drive and set data paths
from google.colab import drive
import os

drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA = '/content/drive/MyDrive/antenna_dataset'

# Create necessary directories in DATA_ROOT
os.makedirs(f'{DATA_ROOT}/artifacts', exist_ok=True)
os.makedirs(f'{DATA_ROOT}/figures', exist_ok=True)
os.makedirs(f'{DATA_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DATA_ROOT}/checkpoints/scaleup', exist_ok=True)
os.makedirs(f'{DATA_ROOT}/checkpoints/tradeoff', exist_ok=True)
os.makedirs(f'{DATA_ROOT}/checkpoints/stability', exist_ok=True)


---
## Cell A — Setup, Chunk 25 Selection Load & 4,000-Sample Stratified Subset Construction

**Purpose:**
1. Loads frozen models, normalization stats, datasets, and loss weighting vectors.
2. Verifies exact hyperparameter parity with Chunk 22 / Chunk 25 training protocol.
3. Loads `DATA_ROOT/artifacts/asymmetric_selection.json` to get the optimal $\kappa^*$ selected in Chunk 25.
   - If Chunk 25 found no viable $\kappa$ under the 15% MAE cap, halts with a clear error directing the user back to Chunk 25.
4. Constructs the grid-stratified 4,000-sample subset (`tradeoff_subset_4000.json`) from `finetune_pool_indices.json` with `seed=2026` using the exact Chunk 22 stratification function.
5. Performs fast local disk staging (with single-archive tar caching + 16-worker thread pool) and canonical normalization round-trip validation.


In [ ]:
# ==========================================================================
# CELL A — Setup, Selection Load & 4000-Sample Subset Construction
# ==========================================================================

import json
import hashlib
import shutil
import time
import copy
import subprocess
import concurrent.futures
import tarfile
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset as TorchDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             balanced_accuracy_score)
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Disable TF32 for numerical consistency
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
print('TF32 disabled')

SEEDS = [42, 43, 44, 45, 46]
print(f'Sweep Seeds: {SEEDS}')

# ── 1. Hyperparameter Parity Audit ───────────────────────────────────────────
SOURCE_NOTEBOOK_CANDIDATES = [
    f'{DATA_ROOT}/notebooks/chunk25_asymmetric_loss_sweep.ipynb',
    f'{DATA_ROOT}/chunk25_asymmetric_loss_sweep.ipynb',
    f'{REPO_ROOT}/notebooks/chunk25_asymmetric_loss_sweep.ipynb',
    f'{REPO_ROOT}/chunk25_asymmetric_loss_sweep.ipynb',
    f'{DATA_ROOT}/notebooks/chunk_seed_stability.ipynb',
    f'{DATA_ROOT}/chunk_seed_stability.ipynb',
]

SOURCE_NOTEBOOK_PATH = next((p for p in SOURCE_NOTEBOOK_CANDIDATES if os.path.exists(p)), None)
assert SOURCE_NOTEBOOK_PATH is not None, f'Could not locate source notebook in: {SOURCE_NOTEBOOK_CANDIDATES}'
print(f'[INFO] Source notebook for hyperparameter extraction: {SOURCE_NOTEBOOK_PATH}')

with open(SOURCE_NOTEBOOK_PATH, encoding='utf-8') as f:
    _src_nb = json.load(f)

_all_src = '\n'.join(
    ''.join(c.get('source', []))
    for c in _src_nb['cells']
    if c.get('cell_type') == 'code'
)

def _extract_unique(pattern, label, cast=float):
    matches = sorted(set(re.findall(pattern, _all_src)))
    assert len(matches) > 0, f'Could not find match for {label} in {SOURCE_NOTEBOOK_PATH}'
    assert len(matches) == 1, f'Found multiple values for {label} in {SOURCE_NOTEBOOK_PATH}: {matches}'
    return cast(matches[0])

def _extract_from_optimizer_call(param_name, label, cast=float):
    opt_calls = re.findall(r'torch\.optim\.Adam[W]?\((?:[^()]*|\([^()]*\))*\)', _all_src, re.DOTALL)
    if not opt_calls:
        opt_calls = re.findall(r'torch\.optim\.Adam[W]?\([^)]*\)', _all_src, re.DOTALL)
    assert len(opt_calls) > 0, f'No Adam calls found in {SOURCE_NOTEBOOK_PATH}'
    values = set()
    for call in opt_calls:
        m = re.search(rf'\b{param_name}\s*=\s*([0-9.eE+-]+)', call)
        if m:
            values.add(m.group(1))
    assert len(values) == 1, f'Found {len(values)} values for {label}: {values}'
    return cast(values.pop())

train_batch_matches = sorted(set(re.findall(r'(?:train_ds|train_loader)[^;\n]*\bbatch_size\s*=\s*([0-9]+)', _all_src)))
extracted_batch_size = int(train_batch_matches[0]) if len(train_batch_matches) == 1 else _extract_unique(r'\bbatch_size\s*=\s*([0-9]+)', 'batch_size', cast=int)

ORIGINAL_HYPERPARAMS = {
    'lr':             _extract_from_optimizer_call('lr', 'lr'),
    'weight_decay':   _extract_from_optimizer_call('weight_decay', 'weight_decay'),
    'sched_factor':   _extract_unique(r'ReduceLROnPlateau\([^)]*\bfactor\s*=\s*([0-9.eE+-]+)', 'scheduler factor'),
    'sched_patience': _extract_unique(r'ReduceLROnPlateau\([^)]*\bpatience\s*=\s*([0-9]+)', 'scheduler patience', cast=int),
    'sched_min_lr':   _extract_unique(r'ReduceLROnPlateau\([^)]*\bmin_lr\s*=\s*([0-9.eE+-]+)', 'scheduler min_lr'),
    'max_epochs':     _extract_unique(r'\bMAX_EPOCHS\s*=\s*([0-9]+)', 'MAX_EPOCHS', cast=int),
    'patience':       _extract_unique(r'\bPATIENCE\s*=\s*([0-9]+)', 'PATIENCE', cast=int),
    'max_norm':       _extract_unique(r'\bMAX_NORM\s*=\s*([0-9.eE+-]+)', 'MAX_NORM'),
    'batch_size':     extracted_batch_size,
}

THIS_NOTEBOOK_HYPERPARAMS = {
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'sched_factor': 0.5,
    'sched_patience': 5,
    'sched_min_lr': 1e-6,
    'max_epochs': 60,
    'patience': 10,
    'max_norm': 1.0,
    'batch_size': 128,
}

print('\n' + '=' * 80)
print('HYPERPARAMETER PARITY AUDIT (Source Notebook vs. Chunk 26 Scaleup)')
print('=' * 80)
mismatches = []
for k in ORIGINAL_HYPERPARAMS:
    orig_val = ORIGINAL_HYPERPARAMS[k]
    this_val = THIS_NOTEBOOK_HYPERPARAMS.get(k, None)
    match = abs(orig_val - this_val) < 1e-12 if isinstance(orig_val, float) else (orig_val == this_val)
    status = 'MATCH [OK]' if match else 'MISMATCH [FAIL]'
    print(f'{k:<16} | {str(orig_val):<22} | {str(this_val):<22} | {status}')
    if not match:
        mismatches.append((k, orig_val, this_val))

assert len(mismatches) == 0, f'FATAL: Hyperparameter mismatch found: {mismatches}'
print('=' * 80)
print('[PASS] Hyperparameter parity confirmed: training protocol is identical to Chunk 25.')

# ── 2. Load Selected Kappa from Chunk 25 ─────────────────────────────────────
selection_path = f'{DATA_ROOT}/artifacts/asymmetric_selection.json'
assert os.path.exists(selection_path), (
    f'Missing Chunk 25 selection artifact: {selection_path}. '
    f'Run chunk25_asymmetric_loss_sweep.ipynb before running Chunk 26 scaleup.'
)

with open(selection_path) as f:
    selection_record = json.load(f)

selected_kappa = selection_record.get('selected_kappa', None)
verdict_status = selection_record.get('verdict_status', '')

print('\n' + '=' * 80)
print('CHUNK 25 ASYMMETRIC LOSS SELECTION AUDIT')
print('=' * 80)
print(f'Verdict Status: {verdict_status}')
print(f'Selected Kappa: {selected_kappa}')
print(f'Selection Reason: {selection_record.get("selection_reason", "N/A")}')

if verdict_status == 'NO VIABLE ASYMMETRY LEVEL UNDER 15% CAP' or selected_kappa is None or selected_kappa <= 1.0:
    raise RuntimeError(
        f'Chunk 25 found no viable asymmetry level under the 15% MAE cap (status={verdict_status}, '
        f'selected_kappa={selected_kappa}). Scaleup cannot proceed with an unviable kappa. '
        f'Return to Chunk 25 to analyze the Pareto frontier.'
    )

SELECTED_KAPPA = float(selected_kappa)
print(f'[PASS] Active Arms for Scaleup: Arm 1 (kappa = 1.0 Anchor), Arm 2 (kappa = {SELECTED_KAPPA})')
print('=' * 80)

# ── 3. FinetuneDataset & Model Architecture ──────────────────────────────────
class FinetuneDataset(TorchDataset):
    """Fine-tune graph dataset with mandatory z-score normalization at load."""

    def __init__(self, indices, processed_dir_base, s11_mean, s11_std):
        assert s11_mean is not None, 'FinetuneDataset requires s11_mean (got None)'
        assert s11_std is not None,  'FinetuneDataset requires s11_std (got None)'
        self.indices = indices
        self.processed_dir_base = processed_dir_base
        self.s11_mean = s11_mean
        self.s11_std  = s11_std

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        grid_size, local_idx = self.indices[idx]
        path = f'{self.processed_dir_base}/{grid_size}x{grid_size}/sample_{local_idx}.pt'
        data = torch.load(path, weights_only=False)

        data.y_raw = data.y.clone()                              # raw dB
        data.y = (data.y - self.s11_mean) / (self.s11_std + 1e-8)  # normalized
        return data


class GATv2Block(nn.Module):
    def __init__(self, in_channels, out_channels, heads, edge_dim, dropout=0.0):
        super().__init__()
        self.conv = GATv2Conv(
            in_channels, out_channels // heads,
            heads=heads, edge_dim=edge_dim,
            concat=True, dropout=dropout
        )
        self.norm = nn.LayerNorm(out_channels)
        self.residual_proj = (nn.Linear(in_channels, out_channels)
                              if in_channels != out_channels else nn.Identity())
        self.act = nn.ReLU()

    def forward(self, x, edge_index, edge_attr):
        out = self.conv(x, edge_index, edge_attr=edge_attr)
        out = self.norm(out)
        out = self.act(out + self.residual_proj(x))
        return out


class AntennaGNN(nn.Module):
    def __init__(self, node_feat_dim=5, edge_feat_dim=2,
                 hidden_dim=128, heads=8, edge_dim=16,
                 num_blocks=4, output_dim=201,
                 conv_dropout=0.10, mlp_dropout=0.10,
                 dropout_from_block=2):
        super().__init__()
        self.input_proj = nn.Linear(node_feat_dim, hidden_dim)
        self.edge_proj  = nn.Linear(edge_feat_dim, edge_dim)

        self.blocks = nn.ModuleList()
        for i in range(num_blocks):
            block_dropout = conv_dropout if i >= dropout_from_block else 0.0
            self.blocks.append(nn.ModuleList([
                GATv2Block(hidden_dim, hidden_dim, heads, edge_dim, dropout=block_dropout),
                GATv2Block(hidden_dim, hidden_dim, heads, edge_dim, dropout=block_dropout),
            ]))

        self.readout_proj = nn.Linear(hidden_dim * 2, 256)
        self.output_mlp = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(mlp_dropout),
            nn.LayerNorm(512),
            nn.Linear(512, output_dim)
        )

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        batch = data.batch

        x = self.input_proj(x)
        edge_attr = self.edge_proj(edge_attr)

        for block in self.blocks:
            for layer in block:
                x = layer(x, edge_index, edge_attr)

        # Metal-only pooling
        metal_mask = data.x[:, 0] > 0.5
        metal_x = x[metal_mask]
        metal_batch = batch[metal_mask]
        pooled = global_mean_pool(metal_x, metal_batch)

        # Virtual node embedding
        virtual_mask = data.x[:, 3] == -1
        virtual_x = x[virtual_mask]

        combined = torch.cat([pooled, virtual_x], dim=-1)
        out = self.readout_proj(combined)
        out = self.output_mlp(out)
        return out


# ── 4. Normalization Statistics & Frequency Weighting ─────────────────────────
freq_axis = np.linspace(1.0, 4.0, 201)

s11_mean_np = np.load(f'{DATA_ROOT}/artifacts/s11_mean.npy')
s11_std_np  = np.load(f'{DATA_ROOT}/artifacts/s11_std.npy')
s11_mean_cpu = torch.tensor(s11_mean_np, dtype=torch.float32)
s11_std_cpu  = torch.tensor(s11_std_np,  dtype=torch.float32)
s11_mean_dev = s11_mean_cpu.to(device)
s11_std_dev  = s11_std_cpu.to(device)

std_sq = s11_std_cpu ** 2
w_cpu = std_sq / std_sq.mean()
w_dev = w_cpu.to(device)

# ── 5. Splits & 4000-Sample Stratified Subset Construction ───────────────────
with open(f'{DATA_ROOT}/splits/finetune_pool_indices.json') as f:
    pool_indices = json.load(f)
with open(f'{DATA_ROOT}/splits/finetune_val_indices.json') as f:
    val_indices = json.load(f)

TEST_INDICES_LOADED = False
print(f'Pool indices: {len(pool_indices)}, Validation indices: {len(val_indices)}')

subset_4000_path = f'{DATA_ROOT}/artifacts/tradeoff_subset_4000.json'
SUBSET_DRAW_SEED = 2026

if os.path.exists(subset_4000_path):
    with open(subset_4000_path) as f:
        subset_4000_indices = json.load(f)
    print(f'Loaded existing 4000-sample subset: {len(subset_4000_indices)} samples')
else:
    print(f'Constructing grid-stratified 4,000-sample subset with seed={SUBSET_DRAW_SEED}...')
    rng = np.random.RandomState(SUBSET_DRAW_SEED)
    pool_df = pd.DataFrame(pool_indices, columns=['grid_size', 'local_idx'])

    # Stratified sample proportionally by grid size (verbatim Chunk 22 logic)
    subset_rows = []
    for g, group in pool_df.groupby('grid_size'):
        n_g = len(group)
        n_sample = round(n_g / len(pool_df) * 4000)
        sampled = group.sample(n=n_sample, random_state=rng)
        subset_rows.append(sampled)

    subset_df = pd.concat(subset_rows)
    if len(subset_df) > 4000:
        subset_df = subset_df.sample(n=4000, random_state=rng)
    elif len(subset_df) < 4000:
        remaining = pool_df.drop(subset_df.index)
        extra = remaining.sample(n=4000 - len(subset_df), random_state=rng)
        subset_df = pd.concat([subset_df, extra])

    subset_4000_indices = subset_df[['grid_size', 'local_idx']].values.tolist()
    with open(subset_4000_path, 'w') as f:
        json.dump(subset_4000_indices, f)
    print(f'Saved new 4000-sample subset to {subset_4000_path}')

assert len(subset_4000_indices) == 4000, f'Expected 4000 subset samples, got {len(subset_4000_indices)}'
SUBSET_HASH_4000 = hashlib.sha256(
    json.dumps(subset_4000_indices, sort_keys=True).encode()).hexdigest()
print(f'Subset 4000 SHA-256 Hash: {SUBSET_HASH_4000[:16]}... (Seed: {SUBSET_DRAW_SEED})')

# Zero overlap assertion with validation set
subset_set = set((gs, li) for gs, li in subset_4000_indices)
val_set    = set((gs, li) for gs, li in val_indices)
overlap_val = subset_set.intersection(val_set)
assert len(overlap_val) == 0, f'FATAL: Overlap detected between 4000-subset and val_indices: {len(overlap_val)}'
print(f'[PASS] Subset isolation confirmed: 0 overlap with validation split ({len(val_indices)} samples).')

# ── 6. Local Disk Staging for Fast I/O ────────────────────────────────────────
processed_dir = f'{DATA_ROOT}/data/processed_finetune'
STAGE_LOCALLY = True
staging_mode = 'Drive-direct'

if STAGE_LOCALLY:
    drive_dir = f'{DATA_ROOT}/data/processed_finetune'
    local_dir = '/content/processed_finetune'
    try:
        needed = set()
        for gs, li in subset_4000_indices + val_indices:
            needed.add((gs, li))
        needed_sorted = sorted(needed)
        n_need = len(needed_sorted)
        print(f'Staging {n_need} unique files (4000 subset + val) to local disk...')

        probe_idx = list(needed)[:20]
        probe_sizes = []
        for gs, li in probe_idx:
            p = f'{drive_dir}/{gs}x{gs}/sample_{li}.pt'
            if os.path.exists(p):
                probe_sizes.append(os.path.getsize(p))
        est_gb = (sum(probe_sizes) / len(probe_sizes)) * n_need / 1e9 if probe_sizes else 0
        free_gb = shutil.disk_usage('/content').free / 1e9
        print(f'~{est_gb:.2f} GB estimated, {free_gb:.1f} GB free on /content')
        assert free_gb > est_gb * 1.5, 'not enough local disk space'

        os.makedirs(local_dir, exist_ok=True)
        needed_hash = hashlib.sha256(
            json.dumps(needed_sorted).encode()).hexdigest()[:16]
        archive_path = f'{DATA_ROOT}/data/_stage_cache/{needed_hash}.tar'

        if os.path.exists(archive_path):
            print(f'Found cached archive for this exact file set: {archive_path}')
            t0 = time.time()
            with tarfile.open(archive_path, 'r') as tf:
                tf.extractall(local_dir)
            print(f'Extracted cached archive in {time.time() - t0:.1f} s '
                  f'(single-file read, no per-sample Drive round-trips)')
            staging_mode = 'cache archive extracted'
        else:
            def _copy_one(item):
                gs, li = item
                src_path = f'{drive_dir}/{gs}x{gs}/sample_{li}.pt'
                dst_dir = f'{local_dir}/{gs}x{gs}'
                dst_path = f'{dst_dir}/sample_{li}.pt'
                if os.path.exists(dst_path):
                    return False
                os.makedirs(dst_dir, exist_ok=True)
                for attempt in range(2):
                    try:
                        shutil.copy2(src_path, dst_path)
                        return True
                    except OSError:
                        if attempt == 0:
                            time.sleep(0.5)
                        else:
                            raise

            t0 = time.time()
            n_copied = 0
            with concurrent.futures.ThreadPoolExecutor(max_workers=16) as ex:
                futures = [ex.submit(_copy_one, item) for item in needed_sorted]
                for fut in tqdm(concurrent.futures.as_completed(futures),
                                total=len(futures), desc='Staging needed files'):
                    if fut.result():
                        n_copied += 1
            print(f'Staged {n_copied} new files in {time.time() - t0:.1f} s '
                  f'(parallel, 16 workers)')

            # Build the cache archive for next time
            try:
                os.makedirs(os.path.dirname(archive_path), exist_ok=True)
                t0_arch = time.time()
                with tarfile.open(archive_path, 'w') as tf:
                    for gs, li in needed_sorted:
                        src_f = f'{local_dir}/{gs}x{gs}/sample_{li}.pt'
                        if os.path.exists(src_f):
                            tf.add(src_f, arcname=f'{gs}x{gs}/sample_{li}.pt')
                print(f'Wrote staging cache archive ({archive_path}) in '
                      f'{time.time() - t0_arch:.1f} s for reuse in future sessions')
            except Exception as arc_e:
                print(f'Warning: failed to write staging cache archive ({arc_e})')

            staging_mode = 'parallel fresh staging + cache built'

        processed_dir = local_dir
    except Exception as e:
        staging_mode = f'fallback to Drive-direct ({type(e).__name__}: {e})'
        print(f'Staging skipped ({type(e).__name__}: {e}); continuing from Drive')

print(f'[STAGING SUMMARY] Mode: {staging_mode} | Processed dir: {processed_dir}')

# ── 7. Datasets & DataLoaders ────────────────────────────────────────────────
train_ds = FinetuneDataset(subset_4000_indices, processed_dir, s11_mean_cpu, s11_std_cpu)
val_ds   = FinetuneDataset(val_indices, processed_dir, s11_mean_cpu, s11_std_cpu)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)

# ── 8. Canonical Normalization Guard ──────────────────────────────────────────
probe_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
ys, yr = [], []
for b in tqdm(probe_loader, desc='Normalization Guard'):
    ys.append(b.y.view(-1, 201))
    yr.append(b.y_raw.view(-1, 201))
all_y, all_raw = torch.cat(ys), torch.cat(yr)
assert not torch.allclose(all_y, all_raw), 'y == y_raw — normalization is a no-op'
recon = all_y * s11_std_cpu + s11_mean_cpu
maxdiff = (recon - all_raw).abs().max().item()
assert maxdiff < 1e-3, f'round trip FAILED: max diff {maxdiff:.6f}'
print(f'[PASS] Normalization guard passed (round trip max diff: {maxdiff:.2e})')
del probe_loader, ys, yr, all_y, all_raw, recon

print('\n[PASS] Cell A complete. Ready for 4000-sample scaleup training.')


---
## Cell B — Scaleup Sweep: 2 Arms ($\kappa=1.0$, $\kappa=\kappa^*$) $\times$ 5 Seeds = 10 Runs

**Protocol:**
1. **Arms:**
   - **Arm 1 ($\kappa = 1.0$ Baseline Anchor):** Chunk 22 frequency-weighted $L_2$ loss on 4,000 samples.
   - **Arm 2 ($\kappa = \kappa^*$ Asymmetric Loss):** Asymmetric penalty $\kappa = \text{selected\_kappa}$ on 4,000 samples.
2. **Seeds:** 5 random seeds $[42, 43, 44, 45, 46]$ for each arm ($2 \times 5 = 10$ runs total).
3. **Training & Architecture:**
   - Warm-start `AntennaGNN` from `DATA_ROOT/checkpoints/best_model.pt`.
   - `Adam(lr=1e-4, weight_decay=1e-4)`, `ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6)`.
   - `MAX_EPOCHS=60, PATIENCE=10, MAX_NORM=1.0, batch_size=128`. (Note: hyperparameters are kept identical to the 2,000-sample setup).
4. **Validation Evaluation:** Evaluated on the identical 1,496 validation samples (`val_loader`). Computes AUROCs, S11 MAE (dB), depth signed bias, depth error, and depth-stratified fixed $-10\text{ dB}$ boundary FNR across all 4 bins.
5. **Checkpoints & CSV:** Checkpoints saved to `DATA_ROOT/checkpoints/scaleup/asym_scaleup_k{kappa}_seed{seed}.pt`. Dataframe saved to `DATA_ROOT/artifacts/asymmetric_scaleup_4000.csv`.


In [ ]:
# ==========================================================================
# CELL B — Scaleup Sweep: 2 Arms x 5 Seeds = 10 Runs
# ==========================================================================

BIN_EDGES = [(-np.inf, -30), (-30, -20), (-20, -15), (-15, -10)]
BIN_LABELS = ['(-inf,-30]', '(-30,-20]', '(-20,-15]', '(-15,-10]']
REPORTABLE_BINS = ['(-15,-10]', '(-20,-15]', '(-30,-20]']

def assign_bin(val):
    for (lo, hi), label in zip(BIN_EDGES, BIN_LABELS):
        if lo < val <= hi:
            return label
    return None

def loss_asym(pred, true, w_f, kappa=1.0):
    """Asymmetric weighted squared error loss."""
    diff = pred - true
    asym_factor = torch.where(diff > 0,
                              torch.as_tensor(kappa, device=diff.device, dtype=diff.dtype),
                              torch.as_tensor(1.0, device=diff.device, dtype=diff.dtype))
    per_point = w_f * (diff ** 2) * asym_factor
    return per_point.mean()

def loss_L2(pred, true):
    """Chunk 22 Arm L2 baseline (standard weighted MSE)."""
    return ((pred - true) ** 2 * w_dev).mean()


# ── Full Evaluation Helper ───────────────────────────────────────────────────
def evaluate_model(model_eval, loader, print_audit=False):
    model_eval.eval()
    records = []
    pos = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred_norm = model_eval(batch)
            pred_db = pred_norm * s11_std_dev + s11_mean_dev
            true_db = batch.y_raw.squeeze(1).to(device)
            pred_np = pred_db.detach().cpu().numpy()
            true_np = true_db.detach().cpu().numpy()

            grids = batch.grid_size
            if isinstance(grids, torch.Tensor):
                grids = grids.tolist()
            is_func_list = batch.is_functioning
            if isinstance(is_func_list, torch.Tensor):
                is_func_list = is_func_list.tolist()

            for i in range(pred_np.shape[0]):
                records.append({
                    'test_idx': pos,
                    'grid': int(grids[i]),
                    'true_min_db': float(true_np[i].min()),
                    'pred_min_db': float(pred_np[i].min()),
                    'is_functioning': bool(is_func_list[i]),
                    's11_mae': float(np.abs(pred_np[i] - true_np[i]).mean()),
                })
                pos += 1

    df = pd.DataFrame(records)
    labels = df['is_functioning'].astype(int).values
    scores = -df['pred_min_db'].values

    # Pooled and 55x55 AUROC
    pooled_auroc = float(roc_auc_score(labels, scores))
    df55 = df[df['grid'] == 55]
    g55_auroc = float(roc_auc_score(df55['is_functioning'].astype(int).values, -df55['pred_min_db'].values))

    # Overall S11 MAE (dB)
    pooled_s11_mae = float(df['s11_mae'].mean())

    # Functioning depth metrics (ground truth true_min_db < -10 dB)
    func_df = df[df['is_functioning']].copy()
    func_df['depth_bin'] = func_df['true_min_db'].apply(assign_bin)

    pooled_depth_bias = float((func_df['pred_min_db'] - func_df['true_min_db']).mean())
    pooled_depth_error = float(np.abs(func_df['pred_min_db'] - func_df['true_min_db']).mean())

    # Depth-stratified breakdown at fixed -10 dB threshold
    bin_counts = {}
    bin_fn = {}
    bin_fnr = {}
    bin_bias = {}

    for bl in BIN_LABELS:
        sub_b = func_df[func_df['depth_bin'] == bl]
        n_b = len(sub_b)
        bin_counts[bl] = n_b
        if n_b > 0:
            fn_b = int((sub_b['pred_min_db'] >= -10.0).sum())
            bin_fn[bl] = fn_b
            bin_fnr[bl] = float(fn_b / n_b)
            bin_bias[bl] = float((sub_b['pred_min_db'] - sub_b['true_min_db']).mean())
        else:
            bin_fn[bl] = 0
            bin_fnr[bl] = float('nan')
            bin_bias[bl] = float('nan')

    # Sample-weighted pooled deep FNR across (-15,-10], (-20,-15], (-30,-20]
    total_fn_deep = sum(bin_fn[b] for b in REPORTABLE_BINS)
    total_n_deep  = sum(bin_counts[b] for b in REPORTABLE_BINS)
    fnr_deep_pooled = float(total_fn_deep / total_n_deep) if total_n_deep > 0 else float('nan')

    # Diagnostic only: swept threshold tau*_bal
    best_tau = -10.0
    best_ba = -1.0
    for tau in np.arange(-5.0, -13.25, -0.25):
        preds = (df['pred_min_db'].values < tau).astype(int)
        ba = balanced_accuracy_score(labels, preds)
        if ba > best_ba:
            best_ba = ba
            best_tau = float(tau)

    if print_audit:
        print('\n' + '-' * 75)
        print('SAMPLE-WEIGHTED POOLED DEEP FNR AUDIT (First Scaleup Evaluation):')
        print('-' * 75)
        print(f'  Bin (-15,-10]: n = {bin_counts["(-15,-10]"]}, FN = {bin_fn["(-15,-10]"]}, FNR = {bin_fnr["(-15,-10]"]:.4f}')
        print(f'  Bin (-20,-15]: n = {bin_counts["(-20,-15]"]}, FN = {bin_fn["(-20,-15]"]}, FNR = {bin_fnr["(-20,-15]"]:.4f}')
        print(f'  Bin (-30,-20]: n = {bin_counts["(-30,-20]"]}, FN = {bin_fn["(-30,-20]"]}, FNR = {bin_fnr["(-30,-20]"]:.4f}')
        print(f'  Tail (-inf,-30] (excluded from scalar summary): n = {bin_counts["(-inf,-30]"]}, FN = {bin_fn["(-inf,-30]"]}, FNR = {bin_fnr["(-inf,-30]"]:.4f}')
        print(f'  POOLED SUM: Total FN = {total_fn_deep} / Total N = {total_n_deep}')
        print(f'  -> fnr_deep_pooled = {total_fn_deep}/{total_n_deep} = {fnr_deep_pooled:.4f}')
        print('-' * 75)

    return {
        'pooled_auroc': pooled_auroc,
        'g55_auroc': g55_auroc,
        'pooled_s11_mae': pooled_s11_mae,
        'pooled_depth_bias': pooled_depth_bias,
        'pooled_depth_error': pooled_depth_error,
        'fnr_15_10': bin_fnr['(-15,-10]'],
        'fnr_20_15': bin_fnr['(-20,-15]'],
        'fnr_30_20': bin_fnr['(-30,-20]'],
        'fnr_inf_30': bin_fnr['(-inf,-30]'],
        'n_15_10': bin_counts['(-15,-10]'],
        'n_20_15': bin_counts['(-20,-15]'],
        'n_30_20': bin_counts['(-30,-20]'],
        'n_inf_30': bin_counts['(-inf,-30]'],
        'fn_15_10': bin_fn['(-15,-10]'],
        'fn_20_15': bin_fn['(-20,-15]'],
        'fn_30_20': bin_fn['(-30,-20]'],
        'fn_inf_30': bin_fn['(-inf,-30]'],
        'fnr_deep_pooled': fnr_deep_pooled,
        'tau_star_bal_diag': best_tau,
    }


# ── Execution Loop (2 Arms x 5 Seeds = 10 Runs) ──────────────────────────────
SCALEUP_ARMS = [1.0, SELECTED_KAPPA]
MAX_EPOCHS = 60
PATIENCE = 10
MAX_NORM = 1.0

all_scaleup_results = []
t0_scaleup_total = time.time()
first_eval = True

print('=' * 85)
print(f'RUNNING 4,000-SAMPLE SCALEUP SWEEP (2 ARMS x 5 SEEDS = 10 RUNS)')
print(f'Arms: kappa = 1.0 (Baseline Anchor) and kappa = {SELECTED_KAPPA} (Selected Asymmetric Loss)')
print('=' * 85)

for kappa in SCALEUP_ARMS:
    arm_label = 'Baseline (kappa=1.0)' if kappa == 1.0 else f'Selected Asym (kappa={kappa})'
    print(f'\n>>> Starting Arm: {arm_label} across 5 seeds...')

    for seed in SEEDS:
        t0_run = time.time()
        print(f'\n[Scaleup 4000] kappa={kappa}, Seed={seed}...')

        torch.manual_seed(seed)
        np.random.seed(seed)

        t_loader = DataLoader(
            train_ds, batch_size=128, shuffle=True,
            num_workers=0, generator=torch.Generator().manual_seed(seed))

        model = AntennaGNN(hidden_dim=128, heads=8, edge_dim=16, num_blocks=4, output_dim=201)
        ckpt_base = torch.load(f'{DATA_ROOT}/checkpoints/best_model.pt', map_location='cpu', weights_only=False)
        model.load_state_dict(ckpt_base['model_state'], strict=True)
        model = model.to(device)
        del ckpt_base

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6)

        best_val_loss = float('inf')
        best_state = None
        best_epoch = -1
        epochs_no_improve = 0

        for epoch in range(MAX_EPOCHS):
            model.train()
            train_loss_acc = 0.0
            n_b = 0
            for batch in t_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                pred = model(batch)
                y = batch.y.squeeze(1)
                l = loss_asym(pred, y, w_dev, kappa=kappa)
                l.backward()
                clip_grad_norm_(model.parameters(), max_norm=MAX_NORM)
                optimizer.step()
                train_loss_acc += l.item()
                n_b += 1

            # Validate
            model.eval()
            val_loss_acc = 0.0
            n_vb = 0
            with torch.no_grad():
                for vbatch in val_loader:
                    vbatch = vbatch.to(device)
                    vpred = model(vbatch)
                    vy = vbatch.y.squeeze(1)
                    vl = loss_asym(vpred, vy, w_dev, kappa=kappa)
                    val_loss_acc += vl.item()
                    n_vb += 1

            val_loss = val_loss_acc / n_vb
            scheduler.step(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                best_epoch = epoch + 1
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= PATIENCE:
                    break

        run_time = time.time() - t0_run

        # Load best weights & evaluate
        model.load_state_dict(best_state)
        eval_metrics = evaluate_model(model, val_loader, print_audit=first_eval)
        first_eval = False

        # Save checkpoint
        save_path = f'{DATA_ROOT}/checkpoints/scaleup/asym_scaleup_k{kappa}_seed{seed}.pt'
        torch.save({
            'model_state': best_state,
            'seed': seed,
            'kappa': kappa,
            'n_train_samples': 4000,
            'subset_hash': SUBSET_HASH_4000,
            'best_epoch': best_epoch,
            'best_val_loss': best_val_loss,
            'eval_metrics': eval_metrics,
            'wall_time_s': run_time,
        }, save_path)

        res_row = {
            'kappa': kappa,
            'seed': seed,
            'n_train_samples': 4000,
            'best_epoch': best_epoch,
            'best_val_loss': best_val_loss,
            'wall_time_s': run_time,
            **eval_metrics
        }
        all_scaleup_results.append(res_row)

        print(f'  Done: kappa={kappa}, Seed={seed} in {run_time/60:.1f}m (Epoch {best_epoch}) | '
              f'AUROC={eval_metrics["pooled_auroc"]:.4f}, 55-AUROC={eval_metrics["g55_auroc"]:.4f}, '
              f'MAE={eval_metrics["pooled_s11_mae"]:.4f} dB, FNR_deep={eval_metrics["fnr_deep_pooled"]:.4f}')

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Save complete dataframe to disk
df_scaleup = pd.DataFrame(all_scaleup_results)
csv_save_path = f'{DATA_ROOT}/artifacts/asymmetric_scaleup_4000.csv'
df_scaleup.to_csv(csv_save_path, index=False)

total_wall_min = (time.time() - t0_scaleup_total) / 60.0
print('\n' + '=' * 85)
print(f'[PASS] Scaleup Sweep (4000 Samples) Complete in {total_wall_min:.1f} min.')
print(f'Saved all 10 rows to {csv_save_path}')
print('=' * 85)


---
## Cell C — Cross-Scale Comparative Analysis (2,000 vs. 4,000 Samples)

**Purpose:** Directly compare the 2,000-sample results (from Chunk 25) against the 4,000-sample results across both arms:
1. **Apples-to-Apples Degradation:** Recomputes $S_{11}$ MAE degradation relative to each scale's own $\kappa=1.0$ anchor:
   $$\text{MAE\_Degradation\_Pct}_{N} = 100 \times \frac{\text{MAE}_{\kappa, N} - \text{MAE}_{1.0, N}}{\text{MAE}_{1.0, N}}$$
2. **Side-by-Side Performance Table:** Compares Pooled AUROC, 55x55 AUROC, S11 MAE (dB), MAE degradation (%), and Mean Deep FNR with 5-seed standard errors.
3. **Comparative 2-Panel Figure:** Visualizes the Pareto frontier shift and depth-stratified FNR changes from $N=2000$ to $N=4000$.
4. **Three-Way Pre-Registered Scaleup Interpretation:** Evaluates whether:
   - (A) Effect holds/strengthens with scale.
   - (B) Effect weakens with scale.
   - (C) Baseline depth under-prediction improves with scale.


In [ ]:
# ==========================================================================
# CELL C — Does More Data Change the Frontier? (Cross-Scale Analysis)
# ==========================================================================

print('=' * 90)
print('CROSS-SCALE COMPARATIVE ANALYSIS: 2,000 vs. 4,000 SAMPLES')
print('=' * 90)

# Load Chunk 25 (2000 samples) and Chunk 26 (4000 samples) results
ch25_csv_path = f'{DATA_ROOT}/artifacts/asymmetric_sweep.csv'
ch26_csv_path = f'{DATA_ROOT}/artifacts/asymmetric_scaleup_4000.csv'

assert os.path.exists(ch25_csv_path), f'Missing {ch25_csv_path}'
assert os.path.exists(ch26_csv_path), f'Missing {ch26_csv_path}'

df_2000_all = pd.read_csv(ch25_csv_path)
df_4000 = pd.read_csv(ch26_csv_path)

# Filter 2000-sample results to the two matching arms (kappa=1.0 and selected_kappa)
df_2000 = df_2000_all[df_2000_all['kappa'].isin([1.0, SELECTED_KAPPA])].copy()
df_2000['n_train_samples'] = 2000

# Recompute per-seed MAE degradation relative to each scale's own kappa=1.0 anchor
# 2000-sample degradation
anchor_2000_by_seed = df_2000[df_2000['kappa'] == 1.0].set_index('seed')['pooled_s11_mae'].to_dict()
df_2000['mae_degradation_pct'] = df_2000.apply(
    lambda r: 100.0 * (r['pooled_s11_mae'] - anchor_2000_by_seed[r['seed']]) / anchor_2000_by_seed[r['seed']],
    axis=1
)

# 4000-sample degradation
anchor_4000_by_seed = df_4000[df_4000['kappa'] == 1.0].set_index('seed')['pooled_s11_mae'].to_dict()
df_4000['mae_degradation_pct'] = df_4000.apply(
    lambda r: 100.0 * (r['pooled_s11_mae'] - anchor_4000_by_seed[r['seed']]) / anchor_4000_by_seed[r['seed']],
    axis=1
)

# ── Summary Aggregation Function ─────────────────────────────────────────────
def summarize_df(df_in, n_samples):
    summary_rows = []
    for k in [1.0, SELECTED_KAPPA]:
        sub = df_in[df_in['kappa'] == k]
        summary_rows.append({
            'n_train_samples': n_samples,
            'kappa': k,
            'arm_label': 'Anchor (κ=1.0)' if k == 1.0 else f'Asym (κ={k:.1f})',
            'mean_mae_db': sub['pooled_s11_mae'].mean(),
            'std_mae_db': sub['pooled_s11_mae'].std(),
            'mean_mae_deg_pct': sub['mae_degradation_pct'].mean(),
            'std_mae_deg_pct': sub['mae_degradation_pct'].std(),
            'mean_fnr_deep': sub['fnr_deep_pooled'].mean(),
            'std_fnr_deep': sub['fnr_deep_pooled'].std(),
            'mean_auroc_pooled': sub['pooled_auroc'].mean(),
            'std_auroc_pooled': sub['pooled_auroc'].std(),
            'mean_auroc_55': sub['g55_auroc'].mean(),
            'std_auroc_55': sub['g55_auroc'].std(),
            'fnr_15_10': sub['fnr_15_10'].mean(),
            'fnr_20_15': sub['fnr_20_15'].mean(),
            'fnr_30_20': sub['fnr_30_20'].mean(),
            'fnr_inf_30': sub['fnr_inf_30'].mean(),
        })
    return pd.DataFrame(summary_rows)

sum_2000 = summarize_df(df_2000, 2000)
sum_4000 = summarize_df(df_4000, 4000)
df_comp_all = pd.concat([sum_2000, sum_4000], ignore_index=True)

# ── Print Direct Side-by-Side Comparison Table ───────────────────────────────
print('\nSide-by-Side Comparison Table (2000 vs. 4000 Training Samples, 5 Seeds):')
print('-' * 115)
print(f'{"N_Samples":<10} | {"Arm":<18} | {"Pooled AUROC":<16} | {"55x55 AUROC":<16} | '
      f'{"S11 MAE (dB)":<16} | {"MAE Deg (%)":<14} | {"Deep FNR":<16}')
print('-' * 115)

for _, r in df_comp_all.iterrows():
    print(f'{int(r["n_train_samples"]):<10} | {r["arm_label"]:<18} | '
          f'{r["mean_auroc_pooled"]:.4f} ± {r["std_auroc_pooled"]:.4f} | '
          f'{r["mean_auroc_55"]:.4f} ± {r["std_auroc_55"]:.4f} | '
          f'{r["mean_mae_db"]:.4f} ± {r["std_mae_db"]:.4f} | '
          f'{r["mean_mae_deg_pct"]:+.2f}% ± {r["std_mae_deg_pct"]:.2f}% | '
          f'{r["mean_fnr_deep"]:.4f} ± {r["std_fnr_deep"]:.4f}')
print('-' * 115)

# ── Compute Key Deltas for Interpretation ────────────────────────────────────
fnr_2000_anchor = sum_2000[sum_2000['kappa'] == 1.0]['mean_fnr_deep'].values[0]
fnr_2000_asym   = sum_2000[sum_2000['kappa'] == SELECTED_KAPPA]['mean_fnr_deep'].values[0]
delta_fnr_2000_pp = (fnr_2000_anchor - fnr_2000_asym) * 100.0

fnr_4000_anchor = sum_4000[sum_4000['kappa'] == 1.0]['mean_fnr_deep'].values[0]
fnr_4000_asym   = sum_4000[sum_4000['kappa'] == SELECTED_KAPPA]['mean_fnr_deep'].values[0]
delta_fnr_4000_pp = (fnr_4000_anchor - fnr_4000_asym) * 100.0

baseline_improvement_pp = (fnr_2000_anchor - fnr_4000_anchor) * 100.0
mae_deg_2000_val = sum_2000[sum_2000['kappa'] == SELECTED_KAPPA]['mean_mae_deg_pct'].values[0]
mae_deg_4000_val = sum_4000[sum_4000['kappa'] == SELECTED_KAPPA]['mean_mae_deg_pct'].values[0]

print(f'\nKey Metrics Summary:')
print(f'  • 2000 Samples: Deep FNR drops from {fnr_2000_anchor:.4f} (κ=1.0) to {fnr_2000_asym:.4f} (κ={SELECTED_KAPPA:.1f}) -> Δ = {delta_fnr_2000_pp:+.2f} pp (MAE deg: {mae_deg_2000_val:+.2f}%)')
print(f'  • 4000 Samples: Deep FNR drops from {fnr_4000_anchor:.4f} (κ=1.0) to {fnr_4000_asym:.4f} (κ={SELECTED_KAPPA:.1f}) -> Δ = {delta_fnr_4000_pp:+.2f} pp (MAE deg: {mae_deg_4000_val:+.2f}%)')
print(f'  • Baseline (κ=1.0) Shift: Deep FNR changes from {fnr_2000_anchor:.4f} (2k) to {fnr_4000_anchor:.4f} (4k) -> Δ = {baseline_improvement_pp:+.2f} pp')

# ── 2-Panel Diagnostic Figure ────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Panel 1: Pareto Frontier Shift (2000 vs 4000 Samples)
colors_scale = {2000: '#1f77b4', 4000: '#2ca02c'}
markers_arm = {1.0: 's', SELECTED_KAPPA: 'o'}

ax1.axvline(15.0, color='red', linestyle='--', linewidth=2.0, label='15% MAE Degradation Cap')
ax1.axvspan(15.0, max(25.0, df_comp_all['mean_mae_deg_pct'].max() + 5.0),
            color='red', alpha=0.08, label='Disallowed (>15% MAE Degradation)')

# Connect 2k points and 4k points
for n in [2000, 4000]:
    sub_n = df_comp_all[df_comp_all['n_train_samples'] == n].sort_values('kappa')
    ax1.plot(sub_n['mean_mae_deg_pct'], sub_n['mean_fnr_deep'],
             color=colors_scale[n], linestyle='-', linewidth=1.8, label=f'Frontier ({n} samples)', zorder=2)

for _, r in df_comp_all.iterrows():
    n = int(r['n_train_samples'])
    k = r['kappa']
    x = r['mean_mae_deg_pct']
    y = r['mean_fnr_deep']
    xerr = r['std_mae_deg_pct']
    yerr = r['std_fnr_deep']
    c = colors_scale[n]
    m = markers_arm[k]

    ax1.errorbar(x, y, xerr=xerr, yerr=yerr, fmt=m, color=c,
                 ecolor=c, elinewidth=2.0, capsize=5, capthick=1.5,
                 markersize=10, zorder=3)

    lbl = f'{n}s: κ={k:.1f}' + (' (Anchor)' if k == 1.0 else ' (Asym)')
    offset = (10, 8) if k == 1.0 else (10, -14)
    ax1.annotate(
        lbl, xy=(x, y), xytext=offset, textcoords='offset points',
        fontsize=10, fontweight='bold', color=c,
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=c, alpha=0.85),
        zorder=4
    )

ax1.set_xlabel('MAE Degradation Relative to Own κ=1.0 Anchor (%)', fontsize=12)
ax1.set_ylabel('Deep Resonance FNR (Fixed -10 dB Boundary, Pooled)', fontsize=12)
ax1.set_title('Pareto Frontier Shift: 2,000 vs. 4,000 Samples', fontsize=13, fontweight='bold')
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc='upper right', fontsize=10)

# Panel 2: Depth-Stratified FNR Across Bins
bin_names = ['(-15,-10]', '(-20,-15]', '(-30,-20]', '(-inf,-30]']
x_bins = np.arange(len(bin_names))
w = 0.20

bar_configs = [
    (2000, 1.0, '#1f77b4', '2k: κ=1.0 Anchor'),
    (2000, SELECTED_KAPPA, '#aec7e8', f'2k: κ={SELECTED_KAPPA:.1f} Asym'),
    (4000, 1.0, '#2ca02c', '4k: κ=1.0 Anchor'),
    (4000, SELECTED_KAPPA, '#98df8a', f'4k: κ={SELECTED_KAPPA:.1f} Asym'),
]

for idx, (n_s, k_val, col, lab) in enumerate(bar_configs):
    r_sub = df_comp_all[(df_comp_all['n_train_samples'] == n_s) & (df_comp_all['kappa'] == k_val)].iloc[0]
    vals = [r_sub['fnr_15_10'], r_sub['fnr_20_15'], r_sub['fnr_30_20'], r_sub['fnr_inf_30']]
    pos = x_bins + (idx - 1.5) * w
    ax2.bar(pos, vals, width=w, color=col, label=lab, edgecolor='black', linewidth=0.5)

ax2.set_xticks(x_bins)
ax2.set_xticklabels(['Shallow
(-15,-10]', 'Moderate
(-20,-15]', 'Deep
(-30,-20]', 'Very Deep Tail
(-inf,-30]'], fontsize=10)
ax2.set_xlabel('Resonance Depth Bin (dB)', fontsize=12)
ax2.set_ylabel('False Negative Rate (FNR @ -10 dB)', fontsize=12)
ax2.set_title('Depth-Stratified FNR Comparison by Data Scale', fontsize=13, fontweight='bold')
ax2.grid(True, linestyle=':', alpha=0.6, axis='y')
ax2.legend(loc='upper right', fontsize=10)

plt.tight_layout()
fig_save_path = f'{DATA_ROOT}/figures/frontier_scaleup_comparison.png'
plt.savefig(fig_save_path, dpi=300, bbox_inches='tight')
plt.show()

print(f'\n[PASS] Saved Scaleup Comparison figure to {fig_save_path}')

# ── 3-Way Pre-Registered Scaleup Interpretation ──────────────────────────────
print('\n' + '=' * 90)
print('THREE-WAY PRE-REGISTERED SCALEUP INTERPRETATION')
print('=' * 90)

verdicts = []

# Interpretation 1 & 2: Asymmetric effect strength across scale
if delta_fnr_4000_pp >= (delta_fnr_2000_pp - 1.5):
    v1 = ('EFFECT HOLDS OR STRENGTHENS WITH SCALE — the asymmetric loss fix is not a '
          f'small-sample artifact (FNR reduction: {delta_fnr_2000_pp:+.2f} pp at 2k vs. {delta_fnr_4000_pp:+.2f} pp at 4k).')
    verdicts.append(v1)
    print(f'[VERDICT 1] {v1}')
else:
    v2 = ('EFFECT WEAKENS WITH SCALE — investigate whether the fix is compensating for an '
          f'under-training regime rather than a structural loss problem (FNR reduction shrank from '
          f'{delta_fnr_2000_pp:+.2f} pp at 2k to {delta_fnr_4000_pp:+.2f} pp at 4k).')
    verdicts.append(v2)
    print(f'[VERDICT 1] {v2}')

# Interpretation 3: Baseline capacity effect
if baseline_improvement_pp >= 2.0:
    v3 = ('BASELINE IMPROVES WITH SCALE — part of the original depth problem may be a '
          f'training-budget effect, separate from the loss-symmetry mechanism; worth testing the '
          f'full 11,971-sample pool as a follow-up, independent of asymmetry (κ=1.0 deep FNR improved '
          f'by {baseline_improvement_pp:+.2f} pp from 2k to 4k).')
    verdicts.append(v3)
    print(f'[VERDICT 2] {v3}')
else:
    v3 = (f'BASELINE STABLE WITH SCALE — symmetric κ=1.0 baseline deep FNR shifted by only '
          f'{baseline_improvement_pp:+.2f} pp between 2k and 4k samples.')
    verdicts.append(v3)
    print(f'[VERDICT 2] {v3}')

print('=' * 90)

# Save scaleup findings to JSON
scaleup_payload = {
    'selected_kappa': SELECTED_KAPPA,
    'delta_fnr_2000_pp': float(delta_fnr_2000_pp),
    'delta_fnr_4000_pp': float(delta_fnr_4000_pp),
    'baseline_improvement_pp': float(baseline_improvement_pp),
    'mae_deg_2000_pct': float(mae_deg_2000_val),
    'mae_deg_4000_pct': float(mae_deg_4000_val),
    'verdicts': verdicts,
    'summary_comparison': df_comp_all.to_dict(orient='records'),
}

scaleup_save_path = f'{DATA_ROOT}/artifacts/scaleup_findings.json'
with open(scaleup_save_path, 'w') as f:
    json.dump(scaleup_payload, f, indent=2)

print(f'Saved scaleup findings record to {scaleup_save_path}')


---
## Cell D — Mandatory Integrity Guards

1. **(a) Loss Function Equivalence:** Re-asserts $\mathcal{L}_{\text{asym}}(\kappa=1.0) \equiv \mathcal{L}_{\text{L2}}$ on real 4,000-sample training batches ($|\Delta| < 10^{-8}$).
2. **(b) Checkpoint Verification:** Asserts that all 10 scaleup checkpoints exist and are non-empty in `DATA_ROOT/checkpoints/scaleup/asym_scaleup_k{kappa}_seed{seed}.pt`.
3. **(c) Subset & Test Isolation:**
   - Asserts `len(subset_4000_indices) == 4000`.
   - Asserts 0 overlap between `subset_4000_indices` and `val_indices`.
   - Asserts `TEST_INDICES_LOADED = False` (test split was never accessed).
4. **(d) Metric Bounds & Validity:** Asserts that all computed metrics fall within valid physical/mathematical bounds.


In [ ]:
# ==========================================================================
# CELL D — Mandatory Integrity Guards
# ==========================================================================

print('=' * 85)
print('MANDATORY INTEGRITY GUARDS (Chunk 26 Scaleup)')
print('=' * 85)

# ── Guard (a): Loss Equivalence on 4000-sample Training Batches ───────────────
print('\nGuard (a): Verifying loss_asym(kappa=1.0) == loss_L2 on train batches...')
t_probe = DataLoader(train_ds, batch_size=32, shuffle=False)
for idx, b in enumerate(t_probe):
    if idx >= 5:
        break
    b = b.to(device)
    y_b = b.y.squeeze(1)
    p_b = y_b + torch.randn_like(y_b) * 0.05
    diff_val = (loss_asym(p_b, y_b, w_dev, kappa=1.0) - loss_L2(p_b, y_b)).abs().item()
    assert diff_val < 1e-8, f'Guard (a) failed on batch {idx}: diff={diff_val:.2e}'

print('[PASS] Guard (a): loss_asym(kappa=1.0) is numerically identical to loss_L2 across all tested batches.')

# ── Guard (b): Check All 10 Scaleup Checkpoints Exist & Non-Empty ─────────────
print('\nGuard (b): Verifying all 10 scaleup checkpoints exist and are non-empty...')
missing_ckpts = []
empty_ckpts = []

for k in SCALEUP_ARMS:
    for s in SEEDS:
        p = f'{DATA_ROOT}/checkpoints/scaleup/asym_scaleup_k{k}_seed{s}.pt'
        if not os.path.exists(p):
            missing_ckpts.append(p)
        elif os.path.getsize(p) == 0:
            empty_ckpts.append(p)

assert len(missing_ckpts) == 0, f'GUARD (b) FAILED: Missing checkpoints: {missing_ckpts}'
assert len(empty_ckpts) == 0, f'GUARD (b) FAILED: Empty checkpoint files: {empty_ckpts}'
print(f'[PASS] Guard (b): All 10 scaleup checkpoints verified present and non-empty in checkpoints/scaleup/.')

# ── Guard (c): Subset & Test Split Isolation Guard ────────────────────────────
print('\nGuard (c): Verifying subset dimensions, hash, and test split isolation...')
assert len(subset_4000_indices) == 4000, f'Expected 4000 subset samples, got {len(subset_4000_indices)}'
subset_set_check = set((gs, li) for gs, li in subset_4000_indices)
val_set_check    = set((gs, li) for gs, li in val_indices)
overlap_check = subset_set_check.intersection(val_set_check)
assert len(overlap_check) == 0, f'GUARD (c) FAILED: Overlap with val set: {len(overlap_check)}'
assert not TEST_INDICES_LOADED, 'GUARD (c) FAILED: test indices were loaded into memory'
print('[PASS] Guard (c): 4000-subset is valid, 0 val overlap, test split isolation strictly maintained.')

# ── Guard (d): Metric Validity & Sanity Checks ────────────────────────────────
print('\nGuard (d): Validating metric bounds in asymmetric_scaleup_4000.csv...')
assert len(df_scaleup) == 10, f'Expected 10 rows in df_scaleup, got {len(df_scaleup)}'
assert (df_scaleup['pooled_auroc'] >= 0.50).all() and (df_scaleup['pooled_auroc'] <= 1.00).all()
assert (df_scaleup['g55_auroc'] >= 0.50).all() and (df_scaleup['g55_auroc'] <= 1.00).all()
assert (df_scaleup['pooled_s11_mae'] > 0.0).all() and (df_scaleup['pooled_s11_mae'] < 10.0).all()
assert (df_scaleup['fnr_deep_pooled'] >= 0.0).all() and (df_scaleup['fnr_deep_pooled'] <= 1.0).all()
print('[PASS] Guard (d): All 10 scaleup run metrics are within valid mathematical and physical ranges.')

print('\n' + '=' * 85)
print('ALL CHUNK 26 GUARDS PASSED [OK]')
print('=' * 85)
